<a href="https://colab.research.google.com/github/cris959/langchain-rag/blob/clase-01/langchain_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-core langchain-community langchain-groq \
  langchain-openai google-genai langchain-google-genai \
  pypdf unstructured bs4 lxml transformers langchain_huggingface \
  sentence_transformers langchain-experimental langchain-pinecone requests==2.32.4


In [ ]:
import os
from google.colab import userdata

# Inyectar credenciales en el entorno
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

print("¡Todas las credenciales listas para RAG Avanzado!")

¡Todas las credenciales listas para RAG Avanzado!


Estructura del Documento: Al usar .load(), LangChain mete todo el texto completo en un solo elemento dentro de una lista. Si el archivo de texto es muy largo, pasárselo así directo a Groq o a tu base de datos de Pinecone va a romper los límites de tokens o arruinará la precisión del RAG.

(Agregando Text Splitting)
En RAG avanzado, inmediatamente después de cargar un documento con TextLoader, debemos fragmentarlo en trozos más pequeños (chunks) que mantengan el contexto semántico.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Cargar el documento de forma segura
loader = TextLoader('/content/mc_beneficios_global.txt', encoding='utf-8')
documento_completo = loader.load()

# 2. Configurar el splitter avanzado
# Definimos chunks de 1000 caracteres con un solapamiento (overlap) de 200
# para que no se pierda el contexto entre fragmentos.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)

# 3. Fragmentar el documento
documentos_fragmentados = text_splitter.split_documents(documento_completo)

# Ver cuántos fragmentos se generaron y revisar el primero
print(f"Total de fragmentos creados: {len(documentos_fragmentados)}")
documentos_fragmentados[0]

Total de fragmentos creados: 20


Document(metadata={'source': '/content/mc_beneficios_global.txt'}, page_content='Guía de Beneficios\nde Mastercard®\npara tarjetas de crédito\nWorld Mastercard emitidas\npor Synchrony Bank')

In [ ]:
# Muestra el objeto Document del final
documentos_fragmentados[-1]

Document(metadata={'source': '/content/mc_beneficios_global.txt'}, page_content='sobre servicios adicionales no descritos en esta Guía. El\nnúmero de teléfono de su institución financiera debe estar\ndisponible en su estado de cuenta mensual o en el reverso de\nsu tarjeta.')

In [ ]:
# Saber cuántos documentos iniciales se cargaron del TextLoader
print(f"Documentos completos cargados: {len(documento_completo)}")

Documentos completos cargados: 1


In [ ]:
len(documento_completo)

1

In [ ]:
# 1. Contemos cuántos caracteres reales tiene tu archivo de texto
print("Caracteres totales en tu archivo:", len(documento_completo[0].page_content))

# 2. Veamos un fragmento del texto real que tienes guardado
print("\nPrimeros 300 caracteres del texto:")
print(documento_completo[0].page_content[:300])

Caracteres totales en tu archivo: 16109

Primeros 300 caracteres del texto:
Guía de Beneficios
de Mastercard®
para tarjetas de crédito
World Mastercard emitidas
por Synchrony Bank

Información importante. Favor leer y guardar.
Esta Guía de beneficios contiene información detallada sobre los
beneficios a los que tiene acceso como tarjetahabiente preferencial.
Esta Guía reemp


El problema es un comportamiento muy específico del RecursiveCharacterTextSplitter. Por defecto, este splitter intenta separar el texto buscando un doble salto de línea (\n\n) y luego saltos de línea simples (\n).
Para obligar a LangChain a que use criterios de corte más flexibles (primero párrafos, luego líneas, luego palabras y finalmente caracteres sueltos), debes pasarle explícitamente el parámetro separators.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuramos el splitter forzando los separadores en orden de prioridad
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=100
)

# Fragmentamos pasando el documento original que cargaste
fragmentos = splitter.split_documents(documento_completo)

# Comprobamos el resultado real
print(f"¡Ahora sí! Total de fragmentos creados: {len(fragmentos)}")

¡Ahora sí! Total de fragmentos creados: 20


In [ ]:
fragmentos[8]


Document(metadata={'source': '/content/mc_beneficios_global.txt'}, page_content='por medio de un texto o una autenticación con base en datos.\nID Theft Protection envía notificaciones de alerta vía correo\nelectrónico, como cuando por ejemplo se presenta un cambio\nde dirección, o en cualquier momento en que se detecten\ninvestigaciones potencialmente no autorizadas o actividades\nsospechosas en el archivo crediticio del consumidor, de modo\nque se puedan tomar medidas inmediatas para minimizar los\ndaños.\nInformación adicional\nSmall Business ID Theft Protection (ID Theft Protection\npara pequeñas empresas) mejora los servicios de ID Theft\nProtection añadiendo el monitoreo de URL y Dominio a\nla lista existente de elementos vigilados. El monitoreo de\nURL y Dominio busca la URL y el Dominio de la empresa del\nconsumidor (limitado a 10 dominios) dentro de las filtraciones\nde datos corporativos, botnets maliciosos de terceros y foros\ncriminales.\n2. RECEPCIÓN DE ALERTAS DE ACTIVIDAD

El modelo gemini-embedding-001 ya quedó deprecado.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Al no pasarle 'api_key', LangChain busca automáticamente 'os.environ["GOOGLE_API_KEY"]'
modelo_embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004")

In [ ]:
print(fragmentos[7].page_content)

las páginas web clandestinas que individuos pueden
visitar sin que terceros puedan rastrear la ubicación de los
visitantes o del editor de la página web. Los sitios de la dark
web constituyen alrededor del .01% de la Internet y están
intencionalmente ocultos o protegidos por tecnologías de
codificación y no son accesibles a través de los navegadores
web estándar.
Información de crédito
Single Bureau Credit Monitoring (Monitoreo del historial
crediticio en una sola agencia) supervisa el archivo crediticio
de TransUnion de un consumidor para identificar cambios
que puedan indicar un fraude, tal como nuevas solicitudes de crédito, un cambio de dirección o nuevas cuentas de
crédito abiertas a su nombre. Para utilizar este servicio, los
consumidores deben proporcionar algunos datos personales,
tal como el nombre, la dirección, la fecha de nacimiento y el
número de Seguro social, y someterse a una verificación digital
por medio de un texto o una autenticación con base en datos.


En el caso del modelo text-embedding-004 de Google, tiene exactamente 768 dimensiones

In [ ]:
from google.colab import userdata

try:
    clave = userdata.get('GEMINI_API_KEY')
    if clave:
        print(f"✅ ¡Clave encontrada en Secrets! Inicia con: {clave[:5]}... y tiene {len(clave)} caracteres.")
    else:
        print("❌ La clave está vacía o no existe en el panel de Secrets.")
except Exception as e:
    print(f"❌ Error al intentar leer Secrets: {e}")

✅ ¡Clave encontrada en Secrets! Inicia con: AQ.Ab... y tiene 53 caracteres.


In [ ]:
# 1. Instalamos la librería necesaria para usar modelos de Hugging Face en LangChain
!pip install -q langchain-community sentence-transformers

from langchain_community.embeddings import HuggingFaceEmbeddings

# 2. Inicializamos el modelo 'all-mpnet-base-v2'
# Este modelo es open-source, súper potente y genera vectores de EXACTAMENTE 768 dimensiones
modelo_embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

# 3. Forzamos la prueba del vector usando tus mismos fragmentos
vector_prueba = modelo_embeddings.embed_query(fragmentos[7].page_content)

print("¡POR FIN! Embeddings generados con éxito de forma local.")
print(f"Dimensiones del vector obtenido: {len(vector_prueba)}")
print("Primeros 5 valores del embedding:", vector_prueba[:5])

/tmp/ipykernel_33734/1884408597.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  modelo_embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

¡POR FIN! Embeddings generados con éxito de forma local.
Dimensiones del vector obtenido: 768
Primeros 5 valores del embedding: [0.005435744300484657, 0.079574353992939, -0.03768467530608177, -0.006786740850657225, -0.012064182199537754]


In [ ]:
# Mide y muestra el tamaño del embedding generado
print(f"La dimensión de mis embeddings es de: {len(modelo_embeddings.embed_query(fragmentos[7].page_content))} dimensiones")

La dimensión de mis embeddings es de: 768 dimensiones


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Configuramos un modelo avanzado que soporta parámetros de dimensiones personalizadas
# o que nativamente escala a altos rangos para RAG complejo
model_kwargs = {'target_size': 3072} # Forzamos la salida a la dimensión del video

modelo_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    encode_kwargs=model_kwargs
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
len(modelo_embeddings.embed_query(fragmentos[7].page_content))

3072

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Inicializamos el modelo base (este nos genera 1024 dimensiones nativas)
modelo_base = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

# 2. Creamos un "adaptador" para forzar las 3072 dimensiones del video
class AdaptadorEmbeddings3072:
    def __init__(self, modelo_interno):
        self.modelo = modelo_interno

    def embed_query(self, text):
        # Obtenemos el vector de 1024
        vector = self.modelo.embed_query(text)
        # Lo rellenamos con ceros hasta llegar a 3072
        return vector + [0.0] * (3072 - len(vector))

    def embed_documents(self, texts):
        # Hace lo mismo para una lista de fragmentos
        return [self.embed_query(t) for t in texts]

# 3. Reemplazamos tu variable oficial con nuestro adaptador de 3072
modelo_embeddings = AdaptadorEmbeddings3072(modelo_base)

/tmp/ipykernel_41916/2218431085.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  modelo_base = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

(El Wrapper Personalizado de 3072)
Vamos a crear una pequeña clase en Python que envuelva al modelo de Hugging Face. Cada vez que LangChain le pida un embedding, este componente generará el vector base de 1024 dimensiones y lo "estirará" matemáticamente con ceros (padding) hasta alcanzar exactamente 3072.

In [ ]:
len(modelo_embeddings.embed_query(fragmentos[7].page_content))

3072

In [ ]:
!pip install -q langchain-pinecone pinecone-client

In [ ]:
# 1. Borramos el paquete viejo que está causando el conflicto
!pip uninstall -y pinecone-client

# 2. Instala los paquetes oficiales actualizados
!pip install -q pinecone langchain-pinecone

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0


In [ ]:
import os
from google.colab import userdata
from langchain_pinecone import PineconeVectorStore

# 1. Traemos la API Key de Pinecone desde tus Secrets de Colab
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

# 2. Definimos el nombre exacto del índice que creaste en tu panel de Pinecone
NOMBRE_INDICE = "mastercard-rag"

# 3. Subimos los documentos usando nuestro adaptador de 3072 dimensiones
print("Subiendo fragmentos a Pinecone... Por favor, espera un momento.")

vectorstore = PineconeVectorStore.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings,
    index_name=NOMBRE_INDICE
)

print("¡Éxito total! Tus vectores ya están alojados en la nube de Pinecone.")

Subiendo fragmentos a Pinecone... Por favor, espera un momento.
¡Éxito total! Tus vectores ya están alojados en la nube de Pinecone.


In [ ]:
# 1. Desinstalamos agresivamente todas las variantes posibles de pinecone
!pip uninstall -y pinecone pinecone-client langchain-pinecone

# 2. Instalamos de manera limpia y exclusiva las versiones modernas requeridas
!pip install -q --no-cache-dir pinecone langchain-pinecone

Found existing installation: pinecone 7.3.0
Uninstalling pinecone-7.3.0:
  Successfully uninstalled pinecone-7.3.0
Found existing installation: langchain-pinecone 0.2.13
Uninstalling langchain-pinecone-0.2.13:
  Successfully uninstalled langchain-pinecone-0.2.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 4.4 MB/s eta 0:00:00


In [ ]:
# 1. Definimos la pregunta que haría el usuario en el bot
query = "¿Cómo debo proceder en el caso de que mi tarjeta haya sido robada?"

# 2. Convertimos la pregunta en embedding usando nuestro adaptador de 3072 dimensiones
query_embed = modelo_embeddings.embed_query(query)

# 3. Comprobamos que mida exactamente lo mismo que el índice de Pinecone
print(f"Dimensión del embedding de la consulta: {len(query_embed)}")

Dimensión del embedding de la consulta: 3072


In [ ]:
# 1. Convertimos nuestra base de datos de Pinecone en un buscador (retriever)
# Configurado para traer los 3 fragmentos más relevantes (top_k = 3)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. Tu línea original: Invocamos la búsqueda pasándole la pregunta en texto plano
fragmentos_similares = retriever.invoke(query)

# 3. Inspeccionamos qué fragmentos de Mastercard encontró en la nube
print(f"¡Se encontraron {len(fragmentos_similares)} fragmentos relevantes!\n")
for i, doc in enumerate(fragmentos_similares):
    print(f"--- Fragmento {i+1} ---")
    print(doc.page_content[:300] + "...") # Mostramos los primeros 300 caracteres

¡Se encontraron 3 fragmentos relevantes!

--- Fragmento 1 ---
sobre servicios adicionales no descritos en esta Guía. El
número de teléfono de su institución financiera debe estar
disponible en su estado de cuenta mensual o en el reverso de
su tarjeta....
--- Fragmento 2 ---
Información importante. Favor leer y guardar.
Esta Guía de beneficios contiene información detallada sobre los
beneficios a los que tiene acceso como tarjetahabiente preferencial.
Esta Guía reemplaza cualquier guía o descripción de programa que
haya recibido anteriormente. Para facilidad de referenc...
--- Fragmento 3 ---
solicitud a los prestamistas de ponerse en contacto con el
consumidor antes de emitir un nuevo crédito. Esto dificulta
que un ladrón de identidad abra nuevas cuentas a su nombre.
Cuando los consumidores colocan una alerta de fraude en una
de las agencias, las otras dos agencias son informadas y la
a...


In [ ]:
# Recorremos los fragmentos que encontramos en Pinecone para mostrarlos ordenados
for i, doc in enumerate(fragmentos_similares):
    print(f"📌 [RESULTADO {i+1}]")
    print(doc.page_content)
    print("-" * 80 + "\n") # Una línea divisoria para que no se mezclen

📌 [RESULTADO 1]
sobre servicios adicionales no descritos en esta Guía. El
número de teléfono de su institución financiera debe estar
disponible en su estado de cuenta mensual o en el reverso de
su tarjeta.
--------------------------------------------------------------------------------

📌 [RESULTADO 2]
Información importante. Favor leer y guardar.
Esta Guía de beneficios contiene información detallada sobre los
beneficios a los que tiene acceso como tarjetahabiente preferencial.
Esta Guía reemplaza cualquier guía o descripción de programa que
haya recibido anteriormente. Para facilidad de referencia, las Guías
de Beneficios para Tarjetahabientes están disponibles tanto en Inglés
como en Español. Sin embargo, tenga en cuenta que para fines
legales y regulatorios, la versión en Inglés de la Guía de Beneficios para
Tarjetahabientes tendrá prioridad de interpretación.
Para presentar una reclamación o para obtener más información
sobre cualquiera de estos servicios, llame al Centro de Asist

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responde utilizando exclusivamente el contenido que se anexa a continuación:\n\nContexto:\n{context}"),
        ("human", "{query}")
    ]
)

In [ ]:
!pip install -q langchain-google-genai

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 1. Traemos tu API Key de Gemini desde los Secrets de Colab
# (Asegúrate de que en la llave 🔑 de Colab tengas guardado 'GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 2. Inicializamos el modelo de lenguaje (LLM) de Google
# Usamos el modelo moderno gemini-2.5-flash con la temperatura que elegiste
modelo = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

# 3. Inicializamos el formateador de salida para recibir texto limpio, no un objeto JSON
parseador = StrOutputParser()

print("¡Modelo Gemini-2.5-Flash configurado con éxito y listo para responder!")

¡Modelo Gemini-2.5-Flash configurado con éxito y listo para responder!


In [ ]:
# 1. Juntamos todos los fragmentos que recuperaste en un solo texto plano
contexto_unificado = "\n\n".join([doc.page_content for doc in fragmentos_similares])

# 2. Le pasamos los datos al prompt para que arme el mensaje estructurado
# (Si en tu prompt usaste {contexto} en vez de {context}, cambia la clave aquí abajo)
mensaje_formateado = prompt.invoke({
    "context": contexto_unificado,
    "query": query
})

# 3. Invocamos al modelo pasándole el mensaje con el contexto de Mastercard
# Usamos el parseador para que la salida sea texto limpio
respuesta_final = (modelo | parseador).invoke(mensaje_formateado)

# 4. Mostramos el resultado en pantalla
print("🤖 RESPUESTA DEL BOT RAG:\n")
print(respuesta_final)

🤖 RESPUESTA DEL BOT RAG:

En el caso de que su tarjeta haya sido robada, el contenido anexo no describe un procedimiento específico para reportar el robo de la tarjeta. Sin embargo, menciona lo siguiente que podría ser relevante:

*   El número de teléfono de su institución financiera debe estar disponible en su estado de cuenta mensual o en el reverso de su tarjeta para servicios adicionales.
*   Para presentar una reclamación o para obtener más información sobre cualquiera de estos servicios, puede llamar al Centro de Asistencia de Mastercard al 1-800-Mastercard (1-800-627-8372), o en Español al: 1-800-633-4466.
*   Puede colocar una alerta de fraude en las agencias de crédito para dificultar que un ladrón de identidad abra nuevas cuentas a su nombre. La activación de estas alertas es gratuita y permanecen en los archivos de crédito durante un año.


In [ ]:
from langchain.globals import set_debug

# 1. Activamos el modo depuración
set_debug(True)

# 2. CONSTRUIMOS LA CADENA: Unimos el prompt, el modelo de Gemini y el parseador de salida
cadena = prompt | modelo | parseador

# 3. Ejecutamos la cadena pasando las variables que tenemos en memoria
respuesta_con_debug = cadena.invoke({
    "query": query,
    "context": contexto_unificado  # Cambia a "contexto" si así lo llamaste en tu ChatPromptTemplate
})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "query": "¿Cómo debo proceder en el caso de que mi tarjeta haya sido robada?",
  "context": "sobre servicios adicionales no descritos en esta Guía. El\nnúmero de teléfono de su institución financiera debe estar\ndisponible en su estado de cuenta mensual o en el reverso de\nsu tarjeta.\n\nInformación importante. Favor leer y guardar.\nEsta Guía de beneficios contiene información detallada sobre los\nbeneficios a los que tiene acceso como tarjetahabiente preferencial.\nEsta Guía reemplaza cualquier guía o descripción de programa que\nhaya recibido anteriormente. Para facilidad de referencia, las Guías\nde Beneficios para Tarjetahabientes están disponibles tanto en Inglés\ncomo en Español. Sin embargo, tenga en cuenta que para fines\nlegales y regulatorios, la versión en Inglés de la Guía de Beneficios para\nTarjetahabientes tendrá prioridad de interpretación.\nPara presentar una reclamación o para obtener más infor